In [1]:
from ultralytics import YOLO
import os
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import cv2

In [ ]:
def polygon_to_mask(polygon_xy, img_height, img_width):
    """
    polygon_xy: list of (x, y) in pixel coordinates
    Returns binary mask of shape (img_height, img_width)
    """
    mask = np.zeros((img_height, img_width), dtype=np.uint8)
    pts = np.array(polygon_xy, dtype=np.int32).reshape((-1, 1, 2))
    cv2.fillPoly(mask, [pts], 1)
    return mask


def parse_seg_label(line, img_width, img_height):
    """
    YOLO seg label: class_id x1 y1 x2 y2 ... xn yn  (normalized)
    Returns (class_id, polygon_xy_pixel, binary_mask)
    """
    parts = line.strip().split()
    if len(parts) < 7:  # class_id + at least 3 points (6 values)
        return None
    class_id = int(float(parts[0]))
    coords = list(map(float, parts[1:]))
    if len(coords) % 2 != 0:
        coords = coords[:-1]
    xs = [c * img_width  for c in coords[0::2]]
    ys = [c * img_height for c in coords[1::2]]
    polygon_xy = list(zip(xs, ys))
    mask = polygon_to_mask(polygon_xy, img_height, img_width)
    return class_id, polygon_xy, mask


def calculate_mask_iou(mask1, mask2):
    """Binary mask IoU"""
    intersection = np.logical_and(mask1, mask2).sum()
    union        = np.logical_or (mask1, mask2).sum()
    return intersection / union if union > 0 else 0.0


def mask_centroid(mask):
    """Returns (cx, cy) centroid of a binary mask"""
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return np.array([0.0, 0.0])
    return np.array([xs.mean(), ys.mean()])

In [ ]:
def get_nodule_records_seg(result, image_path, patient_id, iou_threshold=0.5):
    """
    以 nodule 為單位，回傳一張圖片中每個 nodule 的配對結果（segmentation 版本）。

    每筆 record 包含:
      patient_ID, file_name, match_type ('TP'/'FP'/'FN'),
      pred_polygon, gt_polygon (pixel coords or None),
      iou, conf, center_distance (or None)

    規則:
      - pred nodule 與 GT nodule mask IoU >= threshold  -> TP
      - pred nodule 與 GT maybe_nodule IoU >= threshold -> 忽略
      - pred nodule 無匹配                              -> FP
      - GT nodule 無匹配                               -> FN
    """
    img_height, img_width = result.orig_shape
    file_name = Path(image_path).name

    # --- 預測 masks ---
    pred_masks, pred_polygons, pred_classes, pred_confs = [], [], [], []

    if result.masks is not None and len(result.masks) > 0:
        # masks.data: (N, H, W) float32 on GPU -> binarize
        raw_masks = result.masks.data.cpu().numpy()  # (N, H, W)
        # resize to orig_shape if needed
        for i, m in enumerate(raw_masks):
            if m.shape != (img_height, img_width):
                m = cv2.resize(m, (img_width, img_height), interpolation=cv2.INTER_NEAREST)
            pred_masks.append((m > 0.5).astype(np.uint8))
        pred_polygons = result.masks.xy  # list of np.ndarray (pixel coords)
        pred_classes  = result.boxes.cls.cpu().numpy()
        pred_confs    = result.boxes.conf.cpu().numpy()

    # --- GT 標籤 ---
    label_path = (image_path
                  .replace('/images/', '/labels/')
                  .replace('.png', '.txt')
                  .replace('.jpg', '.txt'))

    gt_masks_nodule,   gt_polygons_nodule   = [], []
    gt_masks_maybe,    gt_polygons_maybe    = [], []

    if Path(label_path).exists():
        with open(label_path, 'r') as f:
            for line in f:
                parsed = parse_seg_label(line, img_width, img_height)
                if parsed is None:
                    continue
                class_id, polygon_xy, mask = parsed
                if class_id == 0:
                    gt_masks_nodule.append(mask)
                    gt_polygons_nodule.append(polygon_xy)
                elif class_id == 1:
                    gt_masks_maybe.append(mask)
                    gt_polygons_maybe.append(polygon_xy)

    pred_nodule_idx = [i for i, cls in enumerate(pred_classes) if cls == 0]

    # --- 匹配 ---
    matched_gt   = set()
    matched_pred = set()
    ignored_pred = set()
    matched_pairs = []  # (pred_idx, gt_idx, iou)

    for i in pred_nodule_idx:
        pred_mask = pred_masks[i]
        best_iou, best_j = 0, -1

        for j, gt_mask in enumerate(gt_masks_nodule):
            if j in matched_gt:
                continue
            iou = calculate_mask_iou(pred_mask, gt_mask)
            if iou > best_iou:
                best_iou, best_j = iou, j

        if best_iou >= iou_threshold:
            matched_gt.add(best_j)
            matched_pred.add(i)
            matched_pairs.append((i, best_j, best_iou))
        else:
            maybe_iou = max(
                (calculate_mask_iou(pred_mask, gm) for gm in gt_masks_maybe),
                default=0
            )
            if maybe_iou >= iou_threshold:
                ignored_pred.add(i)

    # --- 建立 records ---
    records = []
    base = {'patient_ID': patient_id, 'file_name': file_name}

    # TP
    for pred_i, gt_j, iou in matched_pairs:
        pred_poly = pred_polygons[pred_i].tolist() if hasattr(pred_polygons[pred_i], 'tolist') else pred_polygons[pred_i]
        gt_poly   = gt_polygons_nodule[gt_j]
        cd = float(np.linalg.norm(
            mask_centroid(pred_masks[pred_i]) - mask_centroid(gt_masks_nodule[gt_j])
        ))
        records.append({**base,
            'match_type':      'TP',
            'pred_polygon':    pred_poly,
            'gt_polygon':      gt_poly,
            'iou':             iou,
            'conf':            float(pred_confs[pred_i]),
            'center_distance': cd
        })

    # FP
    for i in pred_nodule_idx:
        if i not in matched_pred and i not in ignored_pred:
            pred_poly = pred_polygons[i].tolist() if hasattr(pred_polygons[i], 'tolist') else pred_polygons[i]
            records.append({**base,
                'match_type':      'FP',
                'pred_polygon':    pred_poly,
                'gt_polygon':      None,
                'iou':             0.0,
                'conf':            float(pred_confs[i]),
                'center_distance': None
            })

    # FN
    for j, gt_poly in enumerate(gt_polygons_nodule):
        if j not in matched_gt:
            records.append({**base,
                'match_type':      'FN',
                'pred_polygon':    None,
                'gt_polygon':      gt_poly,
                'iou':             0.0,
                'conf':            None,
                'center_distance': None
            })

    if not records:
        records.append({**base,
            'match_type': 'TN',
            'pred_polygon': None, 'gt_polygon': None,
            'iou': None, 'conf': None, 'center_distance': None
        })

    return records

In [ ]:
def compute_metrics_from_counts(tp, fp, fn):
    eta = 1e-9
    recall    = tp / (tp + fn + eta)
    precision = tp / (tp + fp + eta)
    f1        = 2 * precision * recall / (precision + recall + eta)
    return recall, precision, f1


def get_image_metrics(nodule_df):
    """Image-wise metrics: 每張圖片各自計算 recall / precision / F1"""
    counts = (
        nodule_df[nodule_df['match_type'].isin(['TP', 'FP', 'FN'])]
        .groupby(['file_name', 'match_type'])
        .size()
        .unstack(fill_value=0)
    )
    for col in ['TP', 'FP', 'FN']:
        if col not in counts:
            counts[col] = 0

    counts[['recall', 'precision', 'f1']] = [
        compute_metrics_from_counts(row.TP, row.FP, row.FN)
        for _, row in counts.iterrows()
    ]
    return counts


def get_patient_metrics(nodule_df):
    """Patient-wise metrics: 把同一病患的所有 nodule 合併後計算"""
    counts = (
        nodule_df[nodule_df['match_type'].isin(['TP', 'FP', 'FN'])]
        .groupby(['patient_ID', 'match_type'])
        .size()
        .unstack(fill_value=0)
    )
    for col in ['TP', 'FP', 'FN']:
        if col not in counts:
            counts[col] = 0

    counts[['recall', 'precision', 'f1']] = [
        compute_metrics_from_counts(row.TP, row.FP, row.FN)
        for _, row in counts.iterrows()
    ]
    return counts


def summarize(nodule_df):
    """印出 image-wise 與 patient-wise 的平均指標，以及 TP center distance 統計"""
    img_df = get_image_metrics(nodule_df)
    pat_df = get_patient_metrics(nodule_df)

    print(f"  Image-wise   | recall={img_df['recall'].mean():.4f}  "
          f"precision={img_df['precision'].mean():.4f}  "
          f"F1={img_df['f1'].mean():.4f}")
    print(f"  Patient-wise | recall={pat_df['recall'].mean():.4f}  "
          f"precision={pat_df['precision'].mean():.4f}  "
          f"F1={pat_df['f1'].mean():.4f}")

    tp_dist = nodule_df.loc[nodule_df['match_type'] == 'TP', 'center_distance'].dropna()
    if len(tp_dist) > 0:
        print(f"  Center dist  | n={len(tp_dist)}  "
              f"mean={tp_dist.mean():.2f}  "
              f"median={tp_dist.median():.2f}  "
              f"std={tp_dist.std():.2f}  "
              f"max={tp_dist.max():.2f}")
    else:
        print("  Center dist  | no TP found")

    return img_df, pat_df

In [ ]:
def calculate_ap(nodule_df):
    """
    計算 Average Precision (AP)，使用 101-point interpolation (COCO style)。
    """
    total_gt = nodule_df['match_type'].isin(['TP', 'FN']).sum()
    if total_gt == 0:
        return 0.0, np.array([]), np.array([])

    preds = (
        nodule_df[nodule_df['match_type'].isin(['TP', 'FP'])]
        .dropna(subset=['conf'])
        .sort_values('conf', ascending=False)
        .reset_index(drop=True)
    )

    if len(preds) == 0:
        return 0.0, np.array([0.0]), np.array([0.0])

    tp_cumsum = (preds['match_type'] == 'TP').cumsum().values
    fp_cumsum = (preds['match_type'] == 'FP').cumsum().values

    recalls    = tp_cumsum / total_gt
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum)

    # 101-point interpolation
    ap = 0.0
    for thr in np.linspace(0, 1, 101):
        p_at_thr = precisions[recalls >= thr]
        ap += p_at_thr.max() if len(p_at_thr) > 0 else 0.0
    ap /= 101

    return ap, recalls, precisions


def plot_pr_curve(recalls, precisions, ap, iou_threshold):
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(recalls, precisions, color='steelblue', linewidth=2)
    ax.fill_between(recalls, precisions, alpha=0.2, color='steelblue')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'PR Curve  (IoU={iou_threshold}  AP={ap:.4f})')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.05])
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
name = "2026_03_12(1)"
model_path = f'../yolo_output/runs/seg/{name}/weights/best.pt'
model = YOLO(model_path)

In [ ]:
data_df = pd.read_csv("../yolo_data/all_csv_files/combined_dataset_info_v5.csv")

val_CG_df = data_df[
    (data_df['dataset'] == 'CG') &
    (data_df['train_or_test'] == 'test') &
    (data_df['nodule_type'] != "none")
]

val_df = val_CG_df
data_root = "../yolo_data/all_data_nodule_normal_cut_seg/images"
file_name_list  = [os.path.join(data_root, f"{fn}.png") for fn in val_df['file_name']]
patient_id_list = val_df['patient_ID'].tolist()

print(f"Total images: {len(file_name_list)}")

In [ ]:
import torch

device = 0
print(f"Using device: {device}")

conf_threshold = 0.5
batch_size = 32

# --- Step 1: 推論一次，儲存 raw results ---
raw_results = []  # list of (image_path, patient_id, result)

for i in tqdm(range(0, len(file_name_list), batch_size), desc="Inference"):
    batch_paths = file_name_list[i:i + batch_size]
    batch_pids  = patient_id_list[i:i + batch_size]
    batch_results = model(batch_paths, conf=conf_threshold, verbose=False, device=device)
    for image_path, pid, result in zip(batch_paths, batch_pids, batch_results):
        raw_results.append((image_path, pid, result))

print(f"Inference done. {len(raw_results)} images.")

# --- Step 2: 對每個 IoU threshold 做 matching（不重跑推論）---
iou_thresholds_report = [0.3, 0.5, 0.7]
iou_thresholds_map    = list(np.round(np.arange(0.5, 1.0, 0.05), 2))  # [0.5, 0.55, ..., 0.95]
all_iou_thresholds    = sorted(set(iou_thresholds_report + iou_thresholds_map))

results_by_iou = {}

for iou_threshold in tqdm(all_iou_thresholds, desc="Matching"):
    all_records = []
    for image_path, pid, result in raw_results:
        records = get_nodule_records_seg(result, image_path, pid,
                                         iou_threshold=iou_threshold)
        all_records.extend(records)
    results_by_iou[iou_threshold] = pd.DataFrame(all_records)

# --- 印出 report thresholds 的指標 ---
for iou_threshold in iou_thresholds_report:
    print(f"\n=== IoU threshold = {iou_threshold} ===")
    summarize(results_by_iou[iou_threshold])

In [ ]:
# AP @ IoU 0.3 / 0.5 / 0.7，並畫 PR curve
for iou_threshold in [0.3, 0.5, 0.7]:
    ap, recalls, precisions = calculate_ap(results_by_iou[iou_threshold])
    print(f"AP@IoU={iou_threshold}: {ap:.4f}")
    plot_pr_curve(recalls, precisions, ap, iou_threshold)

# mAP@[0.5:0.95] (COCO style)
aps = [calculate_ap(results_by_iou[iou])[0] for iou in iou_thresholds_map]
print(f"\nmAP@[0.5:0.95]: {np.mean(aps):.4f}")

In [ ]:
# 彙整所有 metric 成一個 DataFrame
rows = []

for iou_threshold in iou_thresholds_report:
    ndf = results_by_iou[iou_threshold]

    ap, _, _ = calculate_ap(ndf)

    img_df = get_image_metrics(ndf)
    img_recall    = img_df['recall'].mean()
    img_precision = img_df['precision'].mean()
    img_f1        = img_df['f1'].mean()

    pat_df = get_patient_metrics(ndf)
    pat_recall    = pat_df['recall'].mean()
    pat_precision = pat_df['precision'].mean()
    pat_f1        = pat_df['f1'].mean()

    tp_dist = ndf.loc[ndf['match_type'] == 'TP', 'center_distance'].dropna()
    cd_mean   = tp_dist.mean()   if len(tp_dist) > 0 else None
    cd_median = tp_dist.median() if len(tp_dist) > 0 else None
    cd_std    = tp_dist.std()    if len(tp_dist) > 0 else None
    cd_max    = tp_dist.max()    if len(tp_dist) > 0 else None

    rows.append({
        'iou_threshold':      iou_threshold,
        'AP':                 round(ap, 4),
        'img_recall':         round(img_recall,    4),
        'img_precision':      round(img_precision, 4),
        'img_f1':             round(img_f1,        4),
        'patient_recall':     round(pat_recall,    4),
        'patient_precision':  round(pat_precision, 4),
        'patient_f1':         round(pat_f1,        4),
        'cd_mean':            round(cd_mean,   2) if cd_mean   is not None else None,
        'cd_median':          round(cd_median, 2) if cd_median is not None else None,
        'cd_std':             round(cd_std,    2) if cd_std    is not None else None,
        'cd_max':             round(cd_max,    2) if cd_max    is not None else None,
    })

# mAP@[0.5:0.95]
map_aps = [calculate_ap(results_by_iou[iou])[0] for iou in iou_thresholds_map]
mAP = round(float(np.mean(map_aps)), 4)

summary_df = pd.DataFrame(rows).set_index('iou_threshold')
summary_df.index.name = 'IoU'

print(f"mAP@[0.5:0.95]: {mAP}")
display(summary_df)

In [ ]:
summary_df.to_csv(f"../yolo_output/runs/seg/{name}/detailed_metrics.csv")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random


def draw_seg_on_image(image_path, records):
    """
    在圖片上畫出 segmentation mask 輪廓。
    顏色規則:
      TP pred mask -> 綠色實線
      TP gt mask   -> 綠色虛線
      FP pred mask -> 紅色實線
      FN gt mask   -> 橘色虛線
    """
    img = cv2.imread(image_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    overlay = img.copy().astype(np.float32)

    fig, ax = plt.subplots(1, 1, figsize=(6, 6))

    color_map = {
        'TP_pred': (0,   255, 0),
        'TP_gt':   (0,   200, 0),
        'FP_pred': (255, 0,   0),
        'FN_gt':   (255, 165, 0),
    }

    def draw_polygon(ax, polygon_xy, color_rgb, linestyle='-', label=None):
        if polygon_xy is None or len(polygon_xy) < 3:
            return
        pts = np.array(polygon_xy)
        xs = np.append(pts[:, 0], pts[0, 0])
        ys = np.append(pts[:, 1], pts[0, 1])
        c = tuple(v / 255 for v in color_rgb)
        ax.plot(xs, ys, color=c, linewidth=2, linestyle=linestyle)

    ax.imshow(img)

    for rec in records:
        mtype = rec['match_type']

        if mtype == 'TP':
            draw_polygon(ax, rec['pred_polygon'], color_map['TP_pred'], '-')
            draw_polygon(ax, rec['gt_polygon'],   color_map['TP_gt'],   '--')
            if rec['pred_polygon'] and len(rec['pred_polygon']) > 0:
                px = rec['pred_polygon'][0][0]
                py = rec['pred_polygon'][0][1]
                cd = rec['center_distance']
                ax.text(px, py - 4, f"TP cd={cd:.1f}", color='lime', fontsize=7)

        elif mtype == 'FP':
            draw_polygon(ax, rec['pred_polygon'], color_map['FP_pred'], '-')
            if rec['pred_polygon'] and len(rec['pred_polygon']) > 0:
                px = rec['pred_polygon'][0][0]
                py = rec['pred_polygon'][0][1]
                ax.text(px, py - 4, "FP", color='red', fontsize=7)

        elif mtype == 'FN':
            draw_polygon(ax, rec['gt_polygon'], color_map['FN_gt'], '--')
            if rec['gt_polygon'] and len(rec['gt_polygon']) > 0:
                px = rec['gt_polygon'][0][0]
                py = rec['gt_polygon'][0][1]
                ax.text(px, py - 4, "FN", color='orange', fontsize=7)

    file_name = records[0]['file_name']
    tp = sum(1 for r in records if r['match_type'] == 'TP')
    fp = sum(1 for r in records if r['match_type'] == 'FP')
    fn = sum(1 for r in records if r['match_type'] == 'FN')
    ax.set_title(f"{file_name}\nTP={tp}  FP={fp}  FN={fn}", fontsize=9)
    ax.axis('off')

    legend_elements = [
        mpatches.Patch(edgecolor='lime',   facecolor='none', linestyle='-',  label='TP pred'),
        mpatches.Patch(edgecolor='green',  facecolor='none', linestyle='--', label='TP gt'),
        mpatches.Patch(edgecolor='red',    facecolor='none', linestyle='-',  label='FP pred'),
        mpatches.Patch(edgecolor='orange', facecolor='none', linestyle='--', label='FN gt'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=7)

    plt.tight_layout()
    return fig


def visualize_samples(nodule_df, data_root, iou_threshold=0.5,
                      n=16, mode='random', seed=42):
    """
    mode:
      'random'  - 隨機抽樣
      'fp'      - FP 最多的圖片
      'fn'      - FN 最多的圖片
      'worst'   - FP+FN 最多的圖片
    """
    grouped = nodule_df.groupby('file_name')

    if mode == 'random':
        rng = random.Random(seed)
        file_names = rng.sample(list(grouped.groups.keys()),
                                min(n, len(grouped)))
    else:
        counts = (
            nodule_df[nodule_df['match_type'].isin(['FP', 'FN'])]
            .groupby(['file_name', 'match_type']).size().unstack(fill_value=0)
        )
        for col in ['FP', 'FN']:
            if col not in counts:
                counts[col] = 0
        if mode == 'fp':
            file_names = counts['FP'].nlargest(n).index.tolist()
        elif mode == 'fn':
            file_names = counts['FN'].nlargest(n).index.tolist()
        else:  # worst
            file_names = (counts['FP'] + counts['FN']).nlargest(n).index.tolist()

    ncols = 4
    nrows = (len(file_names) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
    axes = axes.flatten() if nrows > 1 else [axes] if ncols == 1 else axes.flatten()

    def draw_polygon_ax(ax, polygon_xy, color_rgb, linestyle='-'):
        if polygon_xy is None or len(polygon_xy) < 3:
            return
        pts = np.array(polygon_xy)
        xs = np.append(pts[:, 0], pts[0, 0])
        ys = np.append(pts[:, 1], pts[0, 1])
        c = tuple(v / 255 for v in color_rgb)
        ax.plot(xs, ys, color=c, linewidth=2, linestyle=linestyle)

    for ax_i, fname in enumerate(file_names):
        records = grouped.get_group(fname).to_dict('records')
        image_path = os.path.join(data_root, fname)

        img = cv2.imread(image_path)
        if img is None:
            axes[ax_i].axis('off')
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[ax_i].imshow(img)

        for rec in records:
            mtype = rec['match_type']
            if mtype == 'TP':
                draw_polygon_ax(axes[ax_i], rec['pred_polygon'], (0, 255, 0),   '-')
                draw_polygon_ax(axes[ax_i], rec['gt_polygon'],   (0, 200, 0),   '--')
                if rec['pred_polygon'] and len(rec['pred_polygon']) > 0:
                    px, py = rec['pred_polygon'][0]
                    axes[ax_i].text(px, py - 4,
                                    f"TP cd={rec['center_distance']:.1f}",
                                    color='lime', fontsize=7)
            elif mtype == 'FP':
                draw_polygon_ax(axes[ax_i], rec['pred_polygon'], (255, 0, 0),   '-')
                if rec['pred_polygon'] and len(rec['pred_polygon']) > 0:
                    px, py = rec['pred_polygon'][0]
                    axes[ax_i].text(px, py - 4, "FP", color='red', fontsize=7)
            elif mtype == 'FN':
                draw_polygon_ax(axes[ax_i], rec['gt_polygon'],   (255, 165, 0), '--')
                if rec['gt_polygon'] and len(rec['gt_polygon']) > 0:
                    px, py = rec['gt_polygon'][0]
                    axes[ax_i].text(px, py - 4, "FN", color='orange', fontsize=7)

        tp = sum(1 for r in records if r['match_type'] == 'TP')
        fp = sum(1 for r in records if r['match_type'] == 'FP')
        fn = sum(1 for r in records if r['match_type'] == 'FN')
        axes[ax_i].set_title(f"{fname}\nTP={tp}  FP={fp}  FN={fn}", fontsize=8)
        axes[ax_i].axis('off')

    for ax_i in range(len(file_names), len(axes)):
        axes[ax_i].axis('off')

    legend_elements = [
        mpatches.Patch(edgecolor='lime',   facecolor='none', linestyle='-',  label='TP pred'),
        mpatches.Patch(edgecolor='green',  facecolor='none', linestyle='--', label='TP gt'),
        mpatches.Patch(edgecolor='red',    facecolor='none', linestyle='-',  label='FP pred'),
        mpatches.Patch(edgecolor='orange', facecolor='none', linestyle='--', label='FN gt'),
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=10)
    plt.suptitle(f"IoU={iou_threshold} | mode={mode}", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# 調整 mode 來切換不同視角:
#   'random' - 隨機 16 張
#   'fp'     - FP 最多的 16 張
#   'fn'     - FN 最多的 16 張
#   'worst'  - FP+FN 合計最多的 16 張

iou = 0.3
visualize_samples(results_by_iou[iou], data_root, iou_threshold=iou,
                  n=16, mode='random')